# Who Goes There: evaluate every checkpoint on val and trans_val on Kaggle

Evaluates every checkpoint of one ledger, `LEDGER`, on val and trans_val with the three
evaluation entry points (item 8.08, for the user's item 8.06; `docs/plan.md` section 7):
`docs/ledger.md` by default (phase 1), `docs/ledger-phase2.md` for phase 2's `p2_` runs (item
13.05; `docs/phase2-plan.md`), or `docs/ledger-phase3.md` for phase 3's `p3_` runs (item 13.11;
`docs/phase3-plan.md`). Per checkpoint,
`python -m trailcam.evaluate.evaluate` writes the metrics table and the confusion-matrix,
per-class and per-camera PNGs of both splits, and, for the one run `ASSETS_OVERLAY_RUN` names, the
Grad-CAM overlays of the `assets/` photos; then `python -m trailcam.evaluate.attention` writes the
attention-in-box and animal-removed table and its histogram. Per pair of runs,
`python -m trailcam.evaluate.slices` writes the day/night and common/rare slice tables and the two
feature maps of the two checkpoints side by side. By default, in phase 1, the pair is each run
seed's ResNet-18 arm A and arm B (folder `slices-seed<n>`); in phase 2, each config beside
`p2_baseline` at the same run seed (folder `slices-p2_baseline-<config>-seed<n>`), the pairing of
`docs/phase2-plan.md` section 3; in phase 3 the same with `p3_baseline`, whose one run is seed 0
(`docs/phase3-plan.md` section 2), so only seed 0 is paired by default. `SLICE_PAIRS` names other pairs of configs, each sliced at every
run seed both have. The small CNN gets the first two only. Everything goes under
`/kaggle/working/eval`, one folder per run id and one per pair, and is zipped at the end as
`eval.zip`.

**Phase 2's input size (item 13.05).** Each checkpoint is evaluated at its config's input size:
224 px for every phase-1 config, so phase 1's outputs are as before, and 320 px for `p2_res320`
(shorter side resized to 366, centre crop 320, as `train.py` scored it), for the metrics,
Grad-CAM, attention-in-box and the slices alike. A checkpoint trained at another size than its
config's is refused before any image is read. Phase 2 keeps or drops a change on val and
trans_val alone (`docs/phase2-plan.md` section 3); trans_val is one camera, so `metrics.md`'s
per-class table of trans_val (bobcat recall among it) is read beside its macro-F1, as context
that selects nothing.

This notebook never reads cis_test or trans_test: each entry point accepts val and trans_val
only, and no code cell names a test split or the final scoring. The final scoring (items 9.01 and
9.02) has a notebook of its own and runs once, at the end, with the user's go-ahead (R17). Kept
apart, this notebook can be run as often as needed with no test score in reach, and a Run All
here can never reach one.

Before anything is unpacked or evaluated, section 4 checks every `.pt` file attached against the
`LEDGER` at `COMMIT` by its SHA-256 and refuses a file that matches no row, or a row with no file,
with counts and run ids only. A checkpoint is known by its hash, not its file name, so a renamed
or swapped file cannot be scored under another run's row. Section 4 also checks that each row's
config at `COMMIT` still has the row's config hash, model, arm and split seed, and section 5 that
the cache gives the rows' split hash (R14), so val is the val the runs were selected on.

## How to run on Kaggle

1. Put the checkpoints into one private dataset, once. Each training version kept its checkpoint
   as output, `models/<run id>.pt`. For each of the 7 rows of `docs/ledger.md` (the small CNN, and
   ResNet-18 arm A and arm B for run seeds 0, 1 and 2), open the version of the training notebook
   that printed that run id, then its Output tab, and download `models/<run id>.pt`. Then
   Datasets > New Dataset, title `trailcam-checkpoints`, upload the 7 files, visibility Private.
   Keep the file names (the check goes by the hash, but the run id in a name helps you) and keep
   the original versions: they hold each run's log and the other copy of its checkpoint, and the
   final scoring (item 9.02) needs the same files. A `.pt` file never goes into git (the data
   guard refuses it). For phase 2, a second private dataset, `trailcam-checkpoints-p2`, holds
   the checkpoints of exactly the rows of `docs/ledger-phase2.md` at `COMMIT`, added to as the
   rows are; `CHECKPOINTS_ROOT` then names it (`.../codx888/trailcam-checkpoints-p2`). Phase 3
   does the same with `trailcam-checkpoints-p3` and `docs/ledger-phase3.md`.
2. Import this notebook into Kaggle: File > Import Notebook, with its GitHub link
   (https://github.com/codx999/who-goes-there/blob/main/notebooks/kaggle_evaluate.ipynb).
3. Add Input > Notebook: the output of the packing notebook of item 6.08 (`cct20_512.tar`), as for
   training; `CACHE_ARCHIVE` has the training notebook's default. Add Input > Dataset: the private
   `trailcam-checkpoints`. Kaggle mounts a dataset under `/kaggle/input/datasets/<user>/<slug>/`;
   `CHECKPOINTS_ROOT` is the user's (`codx888`). The `.pt` files are found by searching it at any
   depth; if it holds none, section 4 lists the folders under `/kaggle/input` that hold `.pt`
   files, with counts. Remove the `cct20-512` and `cct20-512-tar` inputs if the notebook has them:
   as in training, section 5 refuses them in an interactive session and warns in a committed one.
4. Keep exactly one parameter cell, the first code cell. Set `COMMIT` to the full sha of the
   `main` commit the coordinator gives. `LEDGER` is `docs/ledger.md` for phase 1 (the default)
   or `docs/ledger-phase2.md` for phase 2, which needs a `COMMIT` with item 13.05's code, or
   `docs/ledger-phase3.md` for phase 3, which needs one with item 13.11's. That
   ledger at `COMMIT` must hold the rows of exactly the attached checkpoints (the default, item
   7.03's commit, holds phase 1's 7), and for overlays its `assets/` must hold the item-8.05
   photos. Set `ASSETS_OVERLAY_RUN` to the run id whose model draws the overlays, or keep `None`
   for none. With no photo under `assets/` at `COMMIT` the notebook makes no overlay and says so;
   everything else runs as usual. Keep `SLICE_PAIRS = None` for the default pairs above, or name
   pairs of the ledger's configs, such as `[("p2_baseline", "p2_res320")]`, or `[]` for no
   slices.
5. Session options: Accelerator GPU T4 x2 (evaluation uses the first T4, as training does) and
   Internet on (the clone and pip).
6. Rehearse: run sections 1 to 5 cell by cell (parameters, clone and install, the code and the
   GPU, the checkpoint check, the unpacking and the cache check; a few minutes), then stop the
   session so no GPU time is idle.
7. Save Version > Save & Run All (Commit). As for training, open the version's Logs tab about 10
   minutes later: if it is still empty and the version still shows "Starting...", cancel it and
   submit again. Each step prints its command line and then its seconds; how long a whole run
   takes is not measured yet.
8. When the version is done, download `eval.zip` from its Output tab and read the summary at the
   end of the log: per run and split, the images, accuracy and macro-F1 now, beside the ledger's.
   Training scored val and trans_val in mixed precision and this notebook scores in float32, so a
   few images near a tie can flip and a score can move in its third or fourth decimal. A
   difference above 0.01 is printed as CHECK: stop and report it before copying anything.
9. Copy these files from `eval.zip` into `docs/figures/`, renamed as shown. The zip's folders are
   run ids: `<S>` is the small CNN's, `<A0>` and `<B0>` seed 0's arm A and arm B, and `<O>` the
   `ASSETS_OVERLAY_RUN`.

   | from `eval.zip` | to `docs/figures/` |
   | --- | --- |
   | `<S>/val_confusion_matrix.png`, `<A0>/...`, `<B0>/...` | `val_confusion_matrix_small_cnn.png`, `val_confusion_matrix_arm_a_seed0.png`, `val_confusion_matrix_arm_b_seed0.png` |
   | `<S>/val_per_class.png`, `<A0>/...`, `<B0>/...` | `val_per_class_small_cnn.png`, `val_per_class_arm_a_seed0.png`, `val_per_class_arm_b_seed0.png` |
   | `<S>/attention_in_box.png`, `<A0>/...`, `<B0>/...` | `attention_val_small_cnn.png`, `attention_val_arm_a_seed0.png`, `attention_val_arm_b_seed0.png` |
   | `slices-seed0/features_by_species.png`, `slices-seed0/features_by_camera.png` | `features_by_species_val_seed0.png`, `features_by_camera_val_seed0.png` |
   | `slices-seed0/slices.md` | `slices_val_seed0.md` |
   | four of `<O>/gradcam_overlay_<nn>.png` | `gradcam_overlay_<nn>.png` |

   That is 15 PNGs and one table. With the 5 data charts of item 8.07 already there, the guard's
   budget for `docs/figures/` (40 files, 20 MB) then holds 20 files, which leaves 20 for the
   trans-test figures of the final scoring (item 9.02). Overlay `<nn>` is the `<nn>`-th photo of
   `assets/` in sorted name order, titled with the prediction and its confidence; the photo's own
   class is in `docs/data-card.md`. Nothing else from the zip goes into the repo: the other files
   stay in the download. Paste the val rows of the summary into `docs/plan.md` section 7 (item
   8.06), then run `uv run python scripts/guard_data.py --all` and `bash scripts/check.sh` before
   the commit. For phase 2, nothing from the zip goes into `docs/figures/`, whose budget is
   phase 1's and the final scoring's: the batch's progress-log entry quotes the summary's rows
   and, from each run's `metrics.md`, trans_val's per-class rows.

Commit this notebook with its outputs cleared (Edit > Clear All Outputs); it prints counts,
scores, camera location numbers, run ids and hashes only, never an image, a file name of the
dataset or a metadata record.

Data: Caltech Camera Traps (CCT20), licence
[CDLA-Permissive 1.0](https://cdla.dev/permissive-1-0/). Please cite: Sara Beery, Grant Van
Horn, Pietro Perona. "Recognition in Terra Incognita." Proceedings of the European Conference
on Computer Vision (ECCV), 2018. https://arxiv.org/abs/1807.04975. Code: MIT.

In [ ]:
# Parameters: the only parameter cell (How to run, step 4); set COMMIT, then Save & Run All.
from pathlib import Path

COMMIT = "b64c48691c4df3f3c2be5acf2cd33a3329104b4e"  # the main commit the coordinator gives
LEDGER = "docs/ledger.md"  # or "docs/ledger-phase2.md" (p2_ runs), "docs/ledger-phase3.md" (p3_)
CACHE_ARCHIVE = Path("/kaggle/input/notebooks/codx888/notebook2b5a4bf6ad/cct20_512.tar")
CHECKPOINTS_ROOT = Path("/kaggle/input/datasets/codx888/trailcam-checkpoints")  # the .pt files
ASSETS_OVERLAY_RUN = None  # None, or the run id whose model draws the assets/ overlays
SLICE_PAIRS = None  # None: the ledger's pairs (How to run, step 4); or [(config, config), ...]

REPO_URL = "https://github.com/codx999/who-goes-there"  # the public repo
INPUT = Path("/kaggle/input")  # where Kaggle mounts the inputs
WORK = Path("/kaggle/working")  # kept as the version's output
EVAL_DIR = WORK / "eval"  # every table and PNG, zipped at the end
CLONE = Path("/tmp/who-goes-there")  # the clone: local disk, not output
CONFIG_FILE = CLONE / "config.yaml"
LEDGER_PATH = CLONE / LEDGER
ASSETS = CLONE / "assets"  # the item-8.05 photos, if COMMIT has them
LOCAL_CACHE = Path("/tmp/trailcam-cache")  # the unpacked archive: local disk, not output

## 1. Check the parameters and pin one GPU

Fails in seconds, before anything is cloned or installed, and prints the parameters in effect (a
second parameter cell would show here): `LEDGER` must be one of the three ledgers, and
`SLICE_PAIRS` `None` or a list of pairs of two different config names. `CUDA_VISIBLE_DEVICES=0` is set here, before torch is
first imported (by the `trailcam` import of section 3), so torch sees the first T4 only.

In [ ]:
import os
import re
import sys

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one T4 of the two; torch is not imported yet
print(f"Python {sys.version.split()[0]}")
if not re.fullmatch(r"[0-9a-f]{40}", COMMIT):
    raise ValueError(f"COMMIT must be a full 40-character commit sha, got {COMMIT!r}")
if LEDGER not in ("docs/ledger.md", "docs/ledger-phase2.md", "docs/ledger-phase3.md"):
    raise ValueError(
        "LEDGER must be docs/ledger.md, docs/ledger-phase2.md or docs/ledger-phase3.md, "
        f"got {LEDGER!r}"
    )
if ASSETS_OVERLAY_RUN is not None and not isinstance(ASSETS_OVERLAY_RUN, str):
    raise ValueError(f"ASSETS_OVERLAY_RUN must be None or a run id, got {ASSETS_OVERLAY_RUN!r}")
if SLICE_PAIRS is not None and not (
    isinstance(SLICE_PAIRS, list)
    and all(
        isinstance(pair, tuple)
        and len(pair) == 2
        and all(isinstance(name, str) for name in pair)
        and pair[0] != pair[1]
        for pair in SLICE_PAIRS
    )
):
    raise ValueError(
        f"SLICE_PAIRS must be None or a list of (config, config) pairs, got {SLICE_PAIRS!r}"
    )
print(f"COMMIT {COMMIT}, LEDGER {LEDGER}, ASSETS_OVERLAY_RUN {ASSETS_OVERLAY_RUN}")
print(f"SLICE_PAIRS {SLICE_PAIRS}")
print(f"CACHE_ARCHIVE {CACHE_ARCHIVE}, CHECKPOINTS_ROOT {CHECKPOINTS_ROOT}")

## 2. Clone the repo at `COMMIT` and install `trailcam` from the clone

`config.yaml`, `docs/ledger.md` and `assets/` are not in the wheel, so they come from this clone:
code, configs, ledger and photos from one commit. This is the project's one `pip install` (the
notebook exception in `CLAUDE.md`), as in the training notebook: pip keeps Kaggle's preinstalled
torch, torchvision, Pillow, timm and PyYAML, and the cell checks that it did. A `COMMIT` without
the code this notebook calls (the archive unpacking of item 6.08, the three entry points of items
8.02 to 8.04, the demo's photo suffixes of item 10.01) is refused before the install; with
`LEDGER = "docs/ledger-phase2.md"`, so is one without item 13.05's evaluation at each config's
input size, and with `LEDGER = "docs/ledger-phase3.md"` one without item 13.11's phase-3 code
(the `p3_` configs, ConvNeXt-Tiny and the phase-3 labels of the slices).

In [ ]:
import subprocess

if CLONE.exists():  # a second run in one interactive session reuses the clone
    print(f"reusing the clone at {CLONE}")
else:
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(CLONE)], check=True)
    subprocess.run(["git", "-C", str(CLONE), "checkout", "--quiet", "--detach", COMMIT], check=True)
head = subprocess.run(
    ["git", "-C", str(CLONE), "rev-parse", "HEAD"], check=True, capture_output=True, text=True
).stdout.strip()
changed = subprocess.run(
    ["git", "-C", str(CLONE), "status", "--porcelain", "--untracked-files=no"],
    check=True,
    capture_output=True,
    text=True,
).stdout.split("\n")
changed = [line for line in changed if line]
if head != COMMIT or changed:
    raise RuntimeError(
        f"{CLONE} is at {head} with {len(changed)} changed files, not a clean {COMMIT}: "
        "restart the session (Run > Factory reset) and run again"
    )
needed_code = {  # what this notebook calls, by module (the item that added it)
    "trailcam/data/cache.py": "def unpack_cache(",  # 6.08
    "trailcam/evaluate/evaluate.py": "def main(",  # 8.02
    "trailcam/evaluate/attention.py": "def main(",  # 8.03
    "trailcam/evaluate/slices.py": "def main(",  # 8.04
    "trailcam/demo/predict.py": "PHOTO_SUFFIXES",  # 10.01
}
if LEDGER == "docs/ledger-phase2.md":  # each config at its input size, pairs by config (13.05)
    needed_code["trailcam/evaluate/slices.py"] = "def model_label("
if LEDGER == "docs/ledger-phase3.md":  # the p3_ configs, ConvNeXt-Tiny, labels by phase (13.11)
    needed_code["trailcam/evaluate/slices.py"] = "phase_of("
lacking = [
    module
    for module, code in needed_code.items()
    if not (CLONE / module).is_file() or code not in (CLONE / module).read_text("utf-8")
]
if lacking:
    raise RuntimeError(
        f"commit {head} lacks {', '.join(lacking)}, which this notebook calls with LEDGER "
        f"{LEDGER}: set COMMIT to the main commit the coordinator gives"
    )
print(f"commit {head}")

In [ ]:
from importlib.metadata import version

KEPT = ("torch", "torchvision", "Pillow", "timm", "PyYAML")  # Kaggle's preinstalled builds
preinstalled = {name: version(name) for name in KEPT}
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", str(CLONE)], check=True)
after_install = {name: version(name) for name in KEPT}
if after_install != preinstalled:
    raise RuntimeError(f"pip replaced preinstalled packages: {preinstalled} -> {after_install}")
print(f"trailcam {version('trailcam')} from commit {head}; kept {after_install}")

## 3. Check the installed code and the GPU

`trailcam` must not be imported yet in this kernel (Python would keep an earlier commit's
modules), and every module of the installed package must match the clone byte for byte.
`check_gpu` refuses a session without a GPU, with other than the one pinned device visible, or
whose device torch was not built for (a P100), so such a session fails here and not at the first
evaluation. The CPU count is printed next to the configs' DataLoader workers, with a warning if
the workers outnumber the CPUs.

In [ ]:
if "trailcam" in sys.modules:  # imported earlier in this kernel: maybe another commit's code
    raise RuntimeError(
        "trailcam is already imported in this kernel and would not be reloaded from the clone: "
        "restart the session (Run > Factory reset) and run again"
    )
import trailcam  # noqa: E402
from trailcam.data.cache import (  # noqa: E402
    archive_files,
    check_images,
    sidecar_folders,
    unpack_cache,
)
from trailcam.data.metadata import DAY_NIGHT_SIDECAR, load_metadata  # noqa: E402
from trailcam.data.split import make_split  # noqa: E402
from trailcam.demo.predict import PHOTO_SUFFIXES  # noqa: E402
from trailcam.evaluate import attention, evaluate, slices  # noqa: E402
from trailcam.evaluate.gradcam import assets_image  # noqa: E402
from trailcam.train.config import config_hash, load_config, load_configs  # noqa: E402
from trailcam.train.device import check_gpu, workers_warning  # noqa: E402
from trailcam.train.ledger import check_ledger, split_cells  # noqa: E402
from trailcam.train.train import CACHE_ANNOTATIONS, CACHE_IMAGES, file_sha256  # noqa: E402

installed = Path(trailcam.__file__).parent
modules = sorted(p.relative_to(CLONE / "trailcam") for p in (CLONE / "trailcam").rglob("*.py"))
differ = [
    str(m) for m in modules if file_sha256(installed / m) != file_sha256(CLONE / "trailcam" / m)
]
if differ:
    raise RuntimeError(f"the installed trailcam differs from the clone in {differ}")
print(f"{len(modules)} trailcam modules at {installed} match commit {head}")

In [ ]:
print(check_gpu())
num_workers = max(c.num_workers for c in load_configs(CONFIG_FILE).values() if not c.fixture)
print(f"{os.cpu_count()} CPUs; the training configs use up to {num_workers} DataLoader workers")
warning = workers_warning(num_workers, os.cpu_count())
if warning:
    print(warning, file=sys.stderr)

## 4. Check every checkpoint against the ledger, before anything is evaluated

The ledger `LEDGER` names, at `COMMIT`, must pass `trailcam.train.ledger`'s check (which checks
`docs/ledger-phase2.md` as phase 2's and `docs/ledger-phase3.md` as phase 3's, by their names).
Every `.pt` file under
`CHECKPOINTS_ROOT`, at any depth, is hashed with SHA-256 and matched to the ledger rows by their
`checkpoint sha256`: a file that matches no row (an unknown checkpoint) and a row with no file (a
missing one) are refused, the unknown ones counted and the missing ones named by run id; so are
two rows of one hash. A file attached twice is one checkpoint. Each row's config at `COMMIT` must
still give the row's config hash, model, arm and split seed. For the slices of section 7: by
default in phase 1, every ResNet-18 run seed needs an arm A and an arm B row; otherwise each
config of a pair (the named ones, or in phase 2 or 3 each config with `p2_baseline` or
`p3_baseline`) must have a row,
neither may be the small CNN, a named pair must share a run seed (a default pair that shares
none is not sliced, and says so), and a config with two rows of one run seed is refused, naming
them, since slices takes one run per config and seed. `ASSETS_OVERLAY_RUN`, if set, must be one
of the ledger's run ids, and the photos directly under `assets/` (none, if `COMMIT` predates item
8.05) are each checked to lie in that folder. All of this runs in seconds, before the cache is
unpacked, and prints run ids, hashes and counts only.

In [ ]:
rows, problems = check_ledger(LEDGER_PATH)
if problems or not rows:
    raise RuntimeError(
        f"{LEDGER} at commit {head} fails its check ({len(problems)} problems, {len(rows)} "
        f"rows): run python -m trailcam.train.ledger --check {LEDGER}"
    )
found = sorted(
    Path(folder) / name
    for folder, _, names in os.walk(CHECKPOINTS_ROOT, followlinks=True)
    for name in names
    if name.endswith(".pt")
)
if not found:
    holding = {
        folder: sum(name.endswith(".pt") for name in names)
        for folder, _, names in os.walk(INPUT, followlinks=True)
        if any(name.endswith(".pt") for name in names)
    }
    raise FileNotFoundError(
        f"no .pt file under CHECKPOINTS_ROOT {CHECKPOINTS_ROOT}; folders under {INPUT} holding "
        f".pt files: {holding or 'none (is the trailcam-checkpoints dataset attached?)'}"
    )
by_digest = {}  # sha256: the first file with it
for path in found:
    by_digest.setdefault(file_sha256(path), path)
ledger_digests = [row["checkpoint sha256"] for row in rows]
unknown = len(set(by_digest) - set(ledger_digests))
missing = [row["run id"] for row in rows if row["checkpoint sha256"] not in by_digest]
shared = len(ledger_digests) - len(set(ledger_digests))
if unknown or missing or shared:
    raise RuntimeError(
        f"the {len(by_digest)} checkpoints under CHECKPOINTS_ROOT do not match the {len(rows)} "
        f"rows of {LEDGER} at commit {head}: {unknown} match no row; {len(missing)} rows "
        f"have no file ({', '.join(missing) or 'none'}); {shared} rows repeat another row's "
        "sha256. Attach exactly the checkpoints of the ledger's rows"
    )
checkpoints = {row["run id"]: by_digest[row["checkpoint sha256"]] for row in rows}
rows_by_run = {row["run id"]: row for row in rows}
print(f"{len(found)} .pt files, {len(by_digest)} distinct, match the {len(rows)} ledger rows:")
pairs = {}  # run seed: {arm: run id} of the ResNet-18 rows, for phase 1's slices
runs_of = {}  # (config, run seed): its run ids, for the named pairs' slices
for row in rows:
    config = load_config(row["config"], CONFIG_FILE)
    cells = {
        "config hash": config_hash(config),
        "model": config.model,
        "arm": config.arm,
        "split seed": str(config.split_seed),
    }
    differs = [column for column, value in cells.items() if value != row[column]]
    if differs:
        raise RuntimeError(
            f"run {row['run id']}: config {row['config']} at commit {head} no longer matches "
            f"its ledger row ({', '.join(differs)})"
        )
    runs_of.setdefault((row["config"], int(row["run seed"])), []).append(row["run id"])
    if row["model"] == "resnet18" and SLICE_PAIRS is None and LEDGER == "docs/ledger.md":
        arms = pairs.setdefault(int(row["run seed"]), {})
        if row["arm"] in arms:
            raise RuntimeError(
                f"runs {arms[row['arm']]} and {row['run id']} are both resnet18 arm {row['arm']} "
                f"of run seed {row['run seed']}: the user keeps one row"
            )
        arms[row["arm"]] = row["run id"]
    print(f"  {row['run id']}: sha256 {row['checkpoint sha256']}")
slice_runs = {}  # output folder: the two run ids slices compares, in that order
if SLICE_PAIRS is None and LEDGER == "docs/ledger.md":  # phase 1: arm A beside arm B, per seed
    unpaired = sorted(seed for seed, arms in pairs.items() if set(arms) != {"A", "B"})
    if unpaired:
        raise RuntimeError(f"run seeds {unpaired} lack an arm A or an arm B row for the slices")
    for seed, arms in sorted(pairs.items()):
        slice_runs[f"slices-seed{seed}"] = [arms["A"], arms["B"]]
    print(f"slices for run seeds {sorted(pairs)}, arm A beside arm B")
else:  # the pairs named, or in phase 2 or 3 every config beside its baseline: same run seed
    configs = list(dict.fromkeys(row["config"] for row in rows))
    named = SLICE_PAIRS
    if named is None:
        base = "p3_baseline" if LEDGER == "docs/ledger-phase3.md" else "p2_baseline"
        named = [(base, name) for name in configs if name != base]
    for first, second in named:
        absent = [name for name in (first, second) if name not in configs]
        if absent:
            raise ValueError(
                f"SLICE_PAIRS: {', '.join(absent)} has no row in {LEDGER} at commit {head}; "
                "name the pairs, or set SLICE_PAIRS = [] for no slices"
            )
        if "small_cnn" in {load_config(name, CONFIG_FILE).model for name in (first, second)}:
            raise ValueError(f"SLICE_PAIRS: ({first}, {second}) holds the small CNN (section 7)")
        seeds = sorted(s for c, s in runs_of if c == first and (second, s) in runs_of)
        if not seeds and SLICE_PAIRS is not None:
            raise RuntimeError(f"SLICE_PAIRS: {first} and {second} share no run seed in {LEDGER}")
        if not seeds:
            print(f"{second} shares no run seed with {first}: not sliced")
        for seed in seeds:
            for name in (first, second):
                if len(runs_of[(name, seed)]) > 1:
                    raise RuntimeError(
                        f"runs {', '.join(runs_of[(name, seed)])} are all {name} of run seed "
                        f"{seed}: slices pairs one run per config and seed"
                    )
            folder = f"slices-{first}-{second}-seed{seed}"
            slice_runs[folder] = [runs_of[(first, seed)][0], runs_of[(second, seed)][0]]
    print(f"slices of {len(slice_runs)} pairs, same run seed: {', '.join(slice_runs) or 'none'}")
if ASSETS_OVERLAY_RUN is not None and ASSETS_OVERLAY_RUN not in checkpoints:
    raise ValueError(
        f"ASSETS_OVERLAY_RUN {ASSETS_OVERLAY_RUN} is not a run id of {LEDGER} at commit {head}"
    )
photos = sorted(
    assets_image(path, ASSETS)
    for path in (ASSETS.iterdir() if ASSETS.is_dir() else [])
    if path.suffix.lower() in PHOTO_SUFFIXES
)
if not photos:
    print("no photo under assets/ at this commit (item 8.05): no Grad-CAM overlays")
elif ASSETS_OVERLAY_RUN is None:
    print(f"{len(photos)} photos under assets/; ASSETS_OVERLAY_RUN is None: no overlays")
else:
    print(f"{len(photos)} photos under assets/: overlays by run {ASSETS_OVERLAY_RUN}")

## 5. Unpack the cache and check it

As in the training notebook (item 6.08): `unpack_cache` extracts the archive into `LOCAL_CACHE`
(under `/tmp`, local disk, not saved as output) and `DATA_ROOT` becomes the unpacked `cct20_512/`
folder; it refuses, before writing anything, any member that is a link or a special file, has an
absolute name or `..`, or lies outside `cct20_512/`. If `CACHE_ARCHIVE` is not a file, the `.tar`
files under `/kaggle/input` are listed. A loose copy of the cache still attached (a folder holding
a `day_night.json` under `/kaggle/input`) is refused in an interactive session and warned about in
a committed one, whose start is already paid for.

The cache must hold the annotations, the images and the day/night sidecar. The metadata is loaded
once and shown as counts only, then `check_images` refuses the cache unless the image of every
kept record of the five splits is under `images/`, naming counts only (R03). Last, the split is
drawn with the ledger's split seed and its hash must be the rows' split hash (R14): otherwise this
cache is not the one the runs were trained and selected on, and val would not be their val.

In [ ]:
loose = sidecar_folders(INPUT)  # the loose 6.03 dataset, attached next to the archive
run_type = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "unknown")  # Interactive or Batch
print(f"run type {run_type}")
if loose:
    attached = (
        f"{len(loose)} folder(s) under {INPUT} hold a {DAY_NIGHT_SIDECAR}: a loose copy of the "
        "cache (the cct20-512 dataset, or the cct20-512-tar one Kaggle extracted) is still "
        "attached, which the archive replaces. Remove it from the inputs"
    )
    if run_type == "Interactive":
        raise RuntimeError(f"{attached}, restart the session and run again")
    print(f"{run_type} run, continuing (its start is paid for): {attached}", file=sys.stderr)
if not CACHE_ARCHIVE.is_file():
    found_archives = [str(path) for path in archive_files(INPUT)]
    raise FileNotFoundError(
        f"CACHE_ARCHIVE {CACHE_ARCHIVE} is not a file; .tar files under {INPUT}: "
        f"{found_archives or 'none (is the output of the packing notebook attached?)'}"
    )
print(f"unpacking {CACHE_ARCHIVE} into {LOCAL_CACHE}", flush=True)
unpacked = unpack_cache(CACHE_ARCHIVE, LOCAL_CACHE)
print(unpacked)
DATA_ROOT = unpacked.root  # the local copy, read by every evaluation below

In [ ]:
needed = {
    "annotations": DATA_ROOT / CACHE_ANNOTATIONS,
    "images": DATA_ROOT / CACHE_IMAGES,
    "day/night sidecar": DATA_ROOT / DAY_NIGHT_SIDECAR,
}
absent = [name for name, path in needed.items() if not path.exists()]
if absent:
    raise FileNotFoundError(f"the unpacked cache {DATA_ROOT} lacks the {', '.join(absent)}")
meta = load_metadata(needed["annotations"], needed["day/night sidecar"])
print(meta)
print(f"{sum(p.is_file() for p in needed['images'].rglob('*'))} files under {CACHE_IMAGES}/")
print(check_images(meta, needed["images"]))
split_seeds = {row["split seed"] for row in rows}
split_hashes = {row["split hash"] for row in rows}
if len(split_seeds) != 1 or len(split_hashes) != 1:
    raise RuntimeError(
        f"the ledger's rows have {len(split_seeds)} split seeds and {len(split_hashes)} split "
        "hashes, not one of each"
    )
split_seed = int(next(iter(split_seeds)))
split_hash = make_split(meta, split_seed).hash
if split_hash not in split_hashes:
    raise RuntimeError(
        f"this cache's split (seed {split_seed}) has hash {split_hash}, the ledger's rows "
        f"{next(iter(split_hashes))}: not the cache the runs were trained on (R14)"
    )
print(f"split seed {split_seed}, split hash {split_hash}: the ledger's")
del meta  # the entry points load the metadata themselves

## 6. Evaluate each checkpoint on val and trans_val

For each ledger row, in ledger order: `trailcam.evaluate.evaluate.main` and then
`trailcam.evaluate.attention.main`, the entry points of `python -m trailcam.evaluate.evaluate` and
`python -m trailcam.evaluate.attention`, with the row's config from the clone's `config.yaml`, the
checkpoint matched in section 4, the local cache, `--split val --split trans_val` and the run's
folder under `EVAL_DIR` as `--out`. Each refuses any other split (R17). The run
`ASSETS_OVERLAY_RUN` names also gets one `--overlay` per photo of `assets/` and `--assets`, so its
folder holds `gradcam_overlay_01.png` onwards, numbered in sorted photo order; no other folder has
an overlay, and the printed command line counts the photos without naming them (a photo's name
may be an image id, R03). A refused argument or a non-zero exit fails the cell. An `EVAL_DIR`
that already holds files (an earlier run in this session) is refused, so no output of two runs is
mixed.

In [ ]:
import time

if EVAL_DIR.exists() and any(EVAL_DIR.iterdir()):
    raise RuntimeError(
        f"{EVAL_DIR} already holds files from an earlier run in this session: restart the "
        "session (Run > Factory reset) and run again"
    )
split_args = ["--split", "val", "--split", "trans_val"]  # R17: never a test split
for run_id, checkpoint in checkpoints.items():
    row = rows_by_run[run_id]
    out = EVAL_DIR / run_id
    data_args = ["--checkpoint", str(checkpoint), "--data-root", str(DATA_ROOT), *split_args]
    argv = ["--config", row["config"], "--config-file", str(CONFIG_FILE), *data_args]
    argv += ["--out", str(out)]
    overlay_args = []
    if run_id == ASSETS_OVERLAY_RUN and photos:
        overlay_args = [arg for photo in photos for arg in ("--overlay", str(photo))]
        overlay_args += ["--assets", str(ASSETS)]
    shown = f" and {len(photos)} --overlay photos of assets/" if overlay_args else ""
    print(f"python -m trailcam.evaluate.evaluate {' '.join(argv)}{shown}", flush=True)
    started = time.perf_counter()
    try:
        exit_code = evaluate.main(argv + overlay_args)
    except SystemExit as stop:  # argparse refused an argument or a path
        raise RuntimeError(
            f"evaluate stopped on run {run_id} (exit {stop.code}), see above"
        ) from None
    if exit_code != 0:
        raise RuntimeError(f"evaluate exited {exit_code} on run {run_id}")
    print(f"evaluate {run_id}: {time.perf_counter() - started:.0f} s", flush=True)

    argv = ["--config-name", row["config"], "--config", str(CONFIG_FILE), *data_args]
    argv += ["--out", str(out)]
    print(f"python -m trailcam.evaluate.attention {' '.join(argv)}", flush=True)
    started = time.perf_counter()
    try:
        exit_code = attention.main(argv)
    except SystemExit as stop:
        raise RuntimeError(
            f"attention stopped on run {run_id} (exit {stop.code}), see above"
        ) from None
    if exit_code != 0:
        raise RuntimeError(f"attention exited {exit_code} on run {run_id}")
    print(f"attention {run_id}: {time.perf_counter() - started:.0f} s", flush=True)
print(f"evaluated {len(checkpoints)} checkpoints on val and trans_val")

## 7. The slices and feature maps, per pair of runs

`trailcam.evaluate.slices.main`, the entry point of `python -m trailcam.evaluate.slices`, once
per pair of section 4, with its two checkpoints in order (so the first is drawn beside the
second), the same splits and the pair's folder under `EVAL_DIR` as `--out`. By default in phase
1 that is once per ResNet-18 run seed, arm A beside arm B, in `slices-seed<n>`; in phase 2,
`p2_baseline` beside each other config at the same run seed, each labelled by its config name
and scored at its own input size; in phase 3 the same with `p3_baseline` (a ConvNeXt-Tiny's
feature map holds its 768 pooled features per image). The small
CNN never reaches it: its features are 25,088 numbers per image, the reason the final scoring
passes `keep_features=False` for it (item 8.04's notes). The command line has no such switch and
keeps the ResNet-18's 512 per image, which the feature maps need, so no call here needs it.

In [ ]:
for folder, compared in slice_runs.items():
    argv = []
    for run_id in compared:
        argv += ["--checkpoint", rows_by_run[run_id]["config"], str(checkpoints[run_id])]
    argv += ["--config", str(CONFIG_FILE), "--data-root", str(DATA_ROOT), *split_args]
    argv += ["--out", str(EVAL_DIR / folder)]
    print(f"python -m trailcam.evaluate.slices {' '.join(argv)}", flush=True)
    started = time.perf_counter()
    try:
        exit_code = slices.main(argv)
    except SystemExit as stop:
        raise RuntimeError(f"slices stopped on {folder} (exit {stop.code}), see above") from None
    if exit_code != 0:
        raise RuntimeError(f"slices exited {exit_code} on {folder}")
    print(f"{folder}: {time.perf_counter() - started:.0f} s", flush=True)

## 8. Zip the outputs and print the summary

`EVAL_DIR` is zipped as `/kaggle/working/eval.zip`, one download for How to run, step 9. The
summary reads each run's `metrics.md` back: per run and split, the images, accuracy and macro-F1
now, beside the ledger's val and trans_val scores of the same checkpoint. Training scored them in
mixed precision and this notebook in float32, so small differences are expected; a score that
differs by more than `tolerance` (0.01) is printed as CHECK, to stop and report before anything is
copied. The scores of two seeds differ by about as much, so the SHA-256 of section 4, not this
comparison, is what pins each checkpoint; the comparison catches a changed pipeline (the split,
the eval transform or the code at `COMMIT`).

In [ ]:
import shutil

files = sorted(p for p in EVAL_DIR.rglob("*") if p.is_file())
archive = Path(shutil.make_archive(str(WORK / "eval"), "zip", EVAL_DIR))
pngs = sum(p.suffix == ".png" for p in files)
tables = sum(p.suffix == ".md" for p in files)
size = archive.stat().st_size / 2**20
print(f"{archive}: {len(files)} files ({pngs} PNG, {tables} tables), {size:.1f} MiB")

In [ ]:
tolerance = 0.01  # above this, a score differs from the ledger by more than float32 explains
print("| run id | split | images | accuracy | macro-F1 | ledger accuracy | ledger macro-F1 |")
print("| --- | --- | --- | --- | --- | --- | --- |")
scored = []
flagged = []
for run_id in checkpoints:
    row = rows_by_run[run_id]
    for line in (EVAL_DIR / run_id / "metrics.md").read_text(encoding="utf-8").splitlines():
        cells = split_cells(line) if line.startswith("|") else []
        if len(cells) == 6 and cells[0] in ("val", "trans_val") and cells[1] in ("cis", "trans"):
            split, _, images, _, now_accuracy, now_f1 = cells
            then_accuracy, then_f1 = row[f"{split} accuracy"], row[f"{split} macro-F1"]
            print(
                f"| {run_id} | {split} | {images} | {now_accuracy} | {now_f1} | {then_accuracy} "
                f"| {then_f1} |"
            )
            scored.append((run_id, split))
            gaps = (float(now_accuracy) - float(then_accuracy), float(now_f1) - float(then_f1))
            if max(abs(gap) for gap in gaps) > tolerance:
                flagged.append(f"{run_id} {split}")
if len(scored) != 2 * len(checkpoints):
    raise RuntimeError(f"found {len(scored)} split rows in the metrics tables, expected 2 per run")
if flagged:
    print(
        f"CHECK: {len(flagged)} of {len(scored)} splits differ from the ledger by more than "
        f"{tolerance}: {', '.join(flagged)}. Stop and report before copying anything",
        file=sys.stderr,
    )
else:
    print(f"all {len(scored)} splits within {tolerance} of the ledger's scores")